# Landslide Detection Pipeline — Stage 1: Data Acquisition (Colab version)

Replicating Shahabi et al. (2024), "Mapping Complex Landslide Scars Using Deep
Learning and High-Resolution Topographic Derivatives from LiDAR Data in
Quebec, Canada," applied to Oregon City / Clackamas County, OR, using USGS
3DEP lidar + DOGAMI SLIDO landslide inventory.

**Run cells top to bottom.** Each section prints output you should look at
before continuing — this notebook is meant to be run interactively, not
executed all-at-once with Runtime → Run all, since a couple of steps need you
to read the printed schema before the next cell's field names will make sense.

No installs needed for this stage — everything here uses `requests`, which
Colab has by default. (PDAL / geospatial libraries come in Stage 2, when we
get to point cloud → DEM.)


## Setup: shared constants


In [ ]:
import requests
import json

SLIDO_BASE = "https://services8.arcgis.com/8PAo5HGmvRMlF2eU/arcgis/rest/services/SLIDO_Release_4p5_wMetadata_gdb/FeatureServer"
DEPOSITS_LAYER = 4
INDEX_LAYERS = {
    "primary": 5,   # Inventory_Map_Index
    "v2": 6,        # Inventory_Map_Index_2
    "v3": 7,        # Inventory_Map_Index_3
}
TNM_API = "https://tnmaccess.nationalmap.gov/api/v1/products"


## Step 1a — Confirm SLIDO Deposits field schema

(Already confirmed as of 2026-07-09 — see comments below — but worth
re-running since ArcGIS services do change over time, and this costs nothing.)


In [ ]:
def get_layer_fields(layer_id):
    url = f"{SLIDO_BASE}/{layer_id}"
    resp = requests.get(url, params={"f": "json"}, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    print(f"Layer name: {data.get('name')}")
    print(f"Geometry type: {data.get('geometryType')}")
    print(f"Object ID field: {data.get('objectIdField')}")
    print(f"\nFields ({len(data.get('fields', []))} total):")
    for f in data.get("fields", []):
        print(f"  {f['name']:30s} {f['type']:35s} alias={f.get('alias')}")
    return data

deposits_schema = get_layer_fields(DEPOSITS_LAYER)


## Step 1b — Download SLIDO Deposits polygons (approximate Oregon City bbox)

This bbox is a **rough estimate**, not a surveyed boundary — Step 2 below
replaces it with DOGAMI's own study-area polygon, which is the better clip
extent since it's the actual footprint the inventory was mapped against.
Use this step for initial exploration only.

Look at the printed `MOVE_CODE` / `MOVE_CLASS` / `CONFIDENCE` / `DEEP_SHAL`
distributions at the bottom — that's the real data your binary-vs-multiclass
decision should be based on.


In [ ]:
# EDIT THIS if you want a different area — see Step 2 for getting the
# real DOGAMI study boundary instead of guessing a bbox.
BBOX_WGS84 = {
    "xmin": -122.6800,
    "ymin": 45.3200,
    "xmax": -122.5400,
    "ymax": 45.4300,
}

def bbox_to_geometry_param(bbox):
    return json.dumps({
        "xmin": bbox["xmin"], "ymin": bbox["ymin"],
        "xmax": bbox["xmax"], "ymax": bbox["ymax"],
        "spatialReference": {"wkid": 4326},
    })

def fetch_deposits(bbox, out_path="slido_deposits_oregon_city.geojson"):
    url = f"{SLIDO_BASE}/{DEPOSITS_LAYER}/query"
    params = {
        "where": "1=1",
        "geometry": bbox_to_geometry_param(bbox),
        "geometryType": "esriGeometryEnvelope",
        "inSR": 4326,
        "spatialRel": "esriSpatialRelIntersects",
        "outFields": "*",
        "returnGeometry": "true",
        "outSR": 4326,
        "f": "geojson",
    }

    all_features = []
    offset = 0
    page_size = 2000  # confirmed Max Record Count for this service

    while True:
        params["resultOffset"] = offset
        params["resultRecordCount"] = page_size
        resp = requests.get(url, params=params, timeout=60)
        resp.raise_for_status()
        data = resp.json()

        if "error" in data:
            raise RuntimeError(f"ArcGIS error: {data['error']}")

        features = data.get("features", [])
        all_features.extend(features)
        print(f"  fetched {len(features)} features (offset={offset}, total so far={len(all_features)})")

        if len(features) < page_size:
            break
        offset += page_size

    fc = {"type": "FeatureCollection", "features": all_features}
    with open(out_path, "w") as fh:
        json.dump(fc, fh)

    print(f"\nSaved {len(all_features)} landslide deposit polygons to {out_path}")
    return all_features

deposits_features = fetch_deposits(BBOX_WGS84)


In [ ]:
# Attribute summary — this decides binary vs multi-class, not a guess made in advance
def summarize_field(features, field):
    counts = {}
    for f in features:
        val = f.get("properties", {}).get(field, "NULL")
        counts[val] = counts.get(val, 0) + 1
    return dict(sorted(counts.items(), key=lambda x: -x[1]))

for field in ["MOVE_CODE", "MOVE_CLASS", "CONFIDENCE", "DEEP_SHAL"]:
    print(f"--- {field} ---")
    for k, v in summarize_field(deposits_features, field).items():
        print(f"  {k}: {v}")
    print()


## Step 2a — Discover the real field schema for DOGAMI's study-area index

I could not confirm the exact field name (e.g. something like `STUDY_NAME`)
ahead of time via search, so this step prints the real schema for all three
index layers. **Read the output before running Step 2b** — you'll need the
actual field name from here.


In [ ]:
def discover_fields(layer_id):
    url = f"{SLIDO_BASE}/{layer_id}"
    resp = requests.get(url, params={"f": "json"}, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    print(f"\n=== Layer {layer_id}: {data.get('name')} ===")
    print(f"Geometry type: {data.get('geometryType')}")
    for f in data.get("fields", []):
        print(f"  {f['name']:30s} {f['type']:30s} alias={f.get('alias')}")
    return data

for key, layer_id in INDEX_LAYERS.items():
    discover_fields(layer_id)


## Step 2b — Get the DOGAMI study boundary polygon(s) intersecting Oregon City

**Update based on the real schema from Step 2a:** there is no place-name
field on this layer. `REF_ID_COD` is confirmed (via DOGAMI's SLIDO structure
docs) to be an **author-citation code** — first 4 letters of the mapper's
last name + initials + a year/sequence suffix (e.g. something like `BURNW23`
for a Bill Burns study) — not a searchable location string. Text-searching
for "Oregon City" against it was never going to work.

Instead, this queries by **geometry** (bbox intersect) — the same pattern
already confirmed working against the Deposits layer — and prints `DATE`,
`LIDAR`, and `SP_42` for whatever comes back, so you can sanity-check it's
the right study/vintage before treating it as your clip boundary. If more
than one polygon intersects the bbox (likely, since studies can overlap or
abut), look at `DATE` to pick the most recent, and `LIDAR`/`SP_42` to
confirm it was mapped using a lidar basemap with DOGAMI's standard protocol.


In [ ]:
# Reuse the same approximate bbox from Step 1b, just as a plain tuple
# (this is still an estimate -- the point of this step is to get DOGAMI's
# real study polygon so you're not stuck using this estimate downstream)
tile_bbox_estimate = (BBOX_WGS84["xmin"], BBOX_WGS84["ymin"], BBOX_WGS84["xmax"], BBOX_WGS84["ymax"])

def fetch_study_polygons(layer_id, bbox):
    url = f"{SLIDO_BASE}/{layer_id}/query"
    params = {
        "where": "1=1",
        "geometry": json.dumps({
            "xmin": bbox[0], "ymin": bbox[1], "xmax": bbox[2], "ymax": bbox[3],
            "spatialReference": {"wkid": 4326},
        }),
        "geometryType": "esriGeometryEnvelope",
        "inSR": 4326,
        "spatialRel": "esriSpatialRelIntersects",
        "outFields": "*",
        "returnGeometry": "true",
        "outSR": 4326,
        "f": "geojson",
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    features = data.get("features", [])
    print(f"Found {len(features)} study polygon(s) intersecting bbox {bbox}\n")
    for feat in features:
        props = feat.get("properties", {})
        print(f"  REF_ID_COD: {props.get('REF_ID_COD')}")
        print(f"    DATE:    {props.get('DATE')}")
        print(f"    LIDAR:   {props.get('LIDAR')}")
        print(f"    SP_42:   {props.get('SP_42')}")
        print(f"    SCALE:   {props.get('SCALE')}")
        print(f"    PURPOSE: {props.get('PURPOSE')}")
        print()

    if features:
        with open("oregon_city_study_boundary.geojson", "w") as fh:
            json.dump({"type": "FeatureCollection", "features": features}, fh)
        print("Saved to oregon_city_study_boundary.geojson")
        if len(features) > 1:
            print("\nMORE THAN ONE STUDY INTERSECTS THIS BBOX.")
            print("This geojson currently contains all of them merged. Before using")
            print("it as a clip boundary, decide whether to keep all, or filter down")
            print("to one REF_ID_COD (edit the cell above to filter by REF_ID_COD")
            print("after this query, or narrow the bbox).")

    return features

study_boundary_features = fetch_study_polygons(INDEX_LAYERS["primary"], tile_bbox_estimate)


**If zero results:** this layer may not have full statewide coverage, or the
bbox needs adjusting. Try the `v2`/`v3` layers (`INDEX_LAYERS["v2"]` /
`["v3"]`) as a fallback, or widen `tile_bbox_estimate` slightly.


## Step 3 — Discover 3DEP lidar tiles covering the study boundary

Uses the study boundary from Step 2b if available; falls back to the
approximate bbox from Step 1b otherwise.

**This step only discovers tiles — it does not download them.** Tiles are
50–500MB each; look at the project name(s) and acquisition date(s) first.

**Important:** DOGAMI mapped SLIDO using DOGAMI/Oregon Lidar Consortium
DEMs, which may be a different survey/vintage than whatever 3DEP project
covers the same footprint. Compare the date(s) printed below against the
source citation in `deposits_schema` / your Step 1b output before assuming
they're the same underlying lidar acquisition. If they differ, that's worth
stating as a limitation in your methods section.


In [ ]:
def bbox_from_geojson_features(features):
    xs, ys = [], []
    for feat in features:
        geom = feat["geometry"]
        coords = geom["coordinates"]
        rings = coords if geom["type"] == "MultiPolygon" else [coords]
        for poly in rings:
            for ring in poly:
                for pt in ring:
                    xs.append(pt[0])
                    ys.append(pt[1])
    return min(xs), min(ys), max(xs), max(ys)

# Use the real study boundary if Step 2b succeeded, else fall back to the bbox
if 'study_boundary_features' in dir() and study_boundary_features:
    tile_bbox = bbox_from_geojson_features(study_boundary_features)
    print(f"Using DOGAMI study boundary bbox: {tile_bbox}")
else:
    tile_bbox = (BBOX_WGS84["xmin"], BBOX_WGS84["ymin"], BBOX_WGS84["xmax"], BBOX_WGS84["ymax"])
    print(f"Falling back to approximate bbox: {tile_bbox}")


In [ ]:
def discover_lidar(bbox, dataset="Lidar Point Cloud (LPC)"):
    # Paginates through ALL matching tiles, not just the first page.
    #
    # IMPORTANT, confirmed via USGS TNM API docs: max + offset combined is
    # capped at 50 by the API itself (not just a default -- a hard ceiling).
    # This was NOT correctly handled in the original version of this cell,
    # which requested max=100 with no offset loop and would have silently
    # truncated results on any area with many matching tiles. Fixed here to
    # page in chunks of 50 and check the response's total field.
    xmin, ymin, xmax, ymax = bbox
    page_size = 50  # documented hard cap for max+offset combined
    all_items = []
    offset = 0
    reported_total = None

    while True:
        params = {
            "datasets": dataset,
            "bbox": f"{xmin},{ymin},{xmax},{ymax}",
            "outputFormat": "JSON",
            "max": page_size,
            "offset": offset,
        }
        resp = requests.get(TNM_API, params=params, timeout=60)
        resp.raise_for_status()
        data = resp.json()

        page_items = data.get("items", [])
        reported_total = data.get("total", reported_total)
        all_items.extend(page_items)
        print(f"  fetched {len(page_items)} (offset={offset}, running total={len(all_items)}"
              f"{f', API reports total={reported_total}' if reported_total is not None else ''})")

        if len(page_items) < page_size:
            break
        offset += page_size
        if offset > 5000:
            print("  WARNING: stopped after 5000 offset without exhausting results.")
            break

    if reported_total is not None and len(all_items) != reported_total:
        print(f"\nWARNING: fetched {len(all_items)} but API reported total={reported_total} -- mismatch, treat as incomplete.")

    items = all_items
    print(f"\nFound {len(items)} lidar product(s) total, intersecting bbox {bbox}\n")

    # NOTE: field names below are CONFIRMED against a real TNM response
    # (checked 2026-07-09) -- sourceId, publicationDate, sizeInBytes,
    # downloadURL, format all exist as shown.
    #
    # IMPORTANT CORRECTION: sourceId is a per-tile ScienceBase catalog ID,
    # NOT a shared project identifier -- grouping by it (an earlier version
    # of this cell did this) puts every tile in its own group of 1, same
    # bug as before just via a different field. The actual shared project
    # identifier is the folder name embedded in downloadURL's path, e.g.
    # ".../Projects/OR_OLCMetro_2019_A19/..." -- extracted via regex below.
    import re
    projects = {}
    for item in items:
        title = item.get("title", "unknown")
        url = item.get("downloadURL", "")
        match = re.search(r'/Projects/([^/]+)/', url)
        key = match.group(1) if match else title  # fallback: no grouping possible

        date = item.get("publicationDate") or item.get("dateCreated") or "unknown date"
        size_mb = item.get("sizeInBytes", 0) / 1e6 if item.get("sizeInBytes") else None
        download_url = item.get("downloadURL")

        projects.setdefault(key, []).append({
            "title": title, "date": date, "size_mb": size_mb, "download_url": download_url,
        })

    print(f"Grouped into {len(projects)} project(s):\n")
    for proj_name, tiles in projects.items():
        print(f"  Project: {proj_name}  ({len(tiles)} tiles)")
        dates = set(t["date"] for t in tiles)
        print(f"    Dates present: {dates}")
        if tiles:
            print(f"    Example tile: {tiles[0]['title']}")
            print(f"    Example URL:  {tiles[0]['download_url']}")
        print()

    with open("3dep_tile_discovery.json", "w") as fh:
        json.dump(items, fh, indent=2)
    print("Full results saved to 3dep_tile_discovery.json")

    return items

lidar_items = discover_lidar(tile_bbox)

# If parsing above looks off, uncomment this to inspect the raw structure:
# print(json.dumps(lidar_items[0], indent=2)) if lidar_items else print("No items returned")


## Downloading files from Colab back to your machine

Everything above writes to Colab's ephemeral local disk (`/content/`), which
is wiped when the runtime disconnects. To keep the files:


In [ ]:
from google.colab import files

# Uncomment the ones you want to keep locally:
# files.download("slido_deposits_oregon_city.geojson")
# files.download("oregon_city_study_boundary.geojson")
# files.download("3dep_tile_discovery.json")


Or, better for a multi-session project, mount Google Drive once at the top
of the notebook and write outputs there directly instead of `/content/`:

```python
from google.colab import drive
drive.mount('/content/drive')
# then use e.g. out_path="/content/drive/MyDrive/landslide_pipeline/slido_deposits.geojson"
```


## What's next (not yet in this notebook)

- Point cloud → bare-earth DEM (via PDAL, recommended against the cloud-native
  EPT data on `s3://usgs-lidar` rather than raw LAZ tiles — OpenTopography
  has ready-made notebooks for this)
- 7 terrain derivatives matching the paper: slope, aspect, hillshade, total
  curvature, TRI, SAR, MDFM
- Rasterize SLIDO Deposits polygons into segmentation masks aligned to the DEM
- 256×256 patch tiling via the GeoPatch package
- Spatially disjoint train/val/test split (2 train / 1 val / 1 test sites)
- ResU-Net (ResNet-50 encoder) + attention gates, PyTorch

Bring back what Steps 1b/2b/3 print for you — the real MOVE_CODE/MOVE_CLASS
counts and whether the 3DEP tile date matches DOGAMI's source lidar — and
we'll use that to settle the binary-vs-multiclass question and pick the
DEM source properly.
